In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import ast

data = pd.read_excel('vitpose_video_consumo_results.xlsx')
df = pd.DataFrame(data)
print(df['delta'].head().to_string())

0    {'cpu_percent': 79.1, 'ram_mb': 541.73046875, ...
1    {'cpu_percent': 71.19999999999999, 'ram_mb': 2...
2    {'cpu_percent': 84.6, 'ram_mb': 0.4453125, 'gp...
3    {'cpu_percent': 82.5, 'ram_mb': 0.390625, 'gpu...
4    {'cpu_percent': 72.2, 'ram_mb': 0.2421875, 'gp...


# Extraer de la columna delta, la informacion de CPU, GPU y RAM

In [2]:
# Verificar si 'delta' es string y convertirlo a diccionario
if isinstance(df['delta'].iloc[0], str):
    try:
        df['delta'] = df['delta'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    except (ValueError, SyntaxError) as e:
        print(f"Error al convertir 'delta': {e}")
        # Si falla, asignar diccionarios vacíos para evitar errores
        df['delta'] = df['delta'].apply(lambda x: {} if isinstance(x, str) else x)

# Crear nuevo DataFrame con las columnas deseadas
nuevo_df = pd.DataFrame()
nuevo_df['threshold'] = df['threshold']  # Copiar la columna original
nuevo_df['device'] = df['device']  # Copiar la columna original

# Extraer 'cpu_percent' y 'ram_mb' del diccionario (manejo seguro con .get())
nuevo_df['cpu_percent'] = df['delta'].apply(lambda x: x.get('cpu_percent', 0))  # 0 si no existe
nuevo_df['ram_mb'] = df['delta'].apply(lambda x: x.get('ram_mb', 0))  # 0 si no existe

# Mostrar el resultado
print(nuevo_df.head())

   threshold device  cpu_percent      ram_mb
0        0.1    cpu         79.1  541.730469
1        0.1    cpu         71.2    2.031250
2        0.1    cpu         84.6    0.445312
3        0.1    cpu         82.5    0.390625
4        0.1    cpu         72.2    0.242188


In [3]:
cpu_stats = nuevo_df.groupby('threshold')['cpu_percent'].describe()
print(cpu_stats.to_string())

            count       mean       std   min   25%   50%     75%   max
threshold                                                             
0.1        6038.0  80.438970  3.547209  37.2  78.1  80.5  83.075  96.1
0.5        6038.0  80.430474  3.684200  43.2  78.1  80.6  83.100  98.3
0.9        6038.0  80.399304  3.426304  36.4  78.1  80.5  82.900  94.2


In [4]:
ram_stats = nuevo_df.groupby('threshold')['ram_mb'].describe()
print(ram_stats.to_string())

            count      mean       std        min    25%       50%       75%         max
threshold                                                                              
0.1        6038.0  0.627946  7.677301 -75.453125  0.125  0.128906  0.132812  541.730469
0.5        6038.0  0.567825  3.002990 -15.542969  0.125  0.128906  0.132812   21.375000
0.9        6038.0  0.571612  2.997925 -15.542969  0.125  0.128906  0.132812   21.378906


In [ ]:
print(nuevo_df['device'].value_counts())

In [ ]:
df_filtrado = nuevo_df[nuevo_df['device'] == 'cpu']
print(df_filtrado['threshold'].value_counts())

# CPU por threshold video

In [ ]:
plt.figure(figsize=(4, 5))  # Tamaño aumentado

# Filtrar el DataFrame para solo filas con 'cpu'
df_filtrado = nuevo_df[nuevo_df['device'] == 'cpu']
cpu_stats = df_filtrado.groupby('threshold')['cpu_percent'].describe()

# Boxplot agrupado por threshold
sns.boxplot(
    data=df_filtrado,
    x='threshold',
    y='cpu_percent',
    palette='Blues',  # Paleta de colores azules
    showmeans=True,
    meanprops={
        'marker': 'o',
        'markerfacecolor': 'white',
        'markeredgecolor': 'black',
        'markersize': 10  # Marcador más grande
    },
    width=0.6  # Ancho de las cajas
)

# Personalización avanzada
# plt.title('Distribución del Uso de CPU por Threshold', fontsize=10, pad=0)
plt.xlabel('Threshold', fontsize=10)
plt.ylabel('Uso de CPU (%)', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Usamos resources_stats para las anotaciones
for i, threshold in enumerate(cpu_stats.index):
    mean_val = cpu_stats.loc[threshold, 'mean']
    median_val = cpu_stats.loc[threshold, '50%']
    std_val = cpu_stats.loc[threshold, 'std']
    count = cpu_stats.loc[threshold, 'count']
    
    plt.text(
        i, 
        df_filtrado['cpu_percent'].max() * 1.1,  # Posición arriba del máximo
        f"n={int(count)}\nμ={mean_val:.2f} ± {std_val:.2f}\nmediana={median_val:.2f}",
        ha='center',
        fontsize=8,
        bbox=dict(
            facecolor='white',
            alpha=0.8,
            edgecolor='gray',
            boxstyle='round,pad=0.5'
        ),
    )

# Línea horizontal de referencia para CPU
# plt.axhline(
#     y=100, 
#     color='red', 
#     linestyle=':', 
#     alpha=0.5, 
#     label='Máximo teórico (CPU)'
# )
# plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

# Uso de RAM por threshold

In [ ]:
plt.figure(figsize=(4, 5))  # Tamaño aumentado

# Boxplot agrupado por threshold
sns.boxplot(
    data=nuevo_df,
    x='threshold',
    y='ram_mb',
    palette='Blues',  # Paleta de colores azules
    showmeans=True,
    meanprops={
        'marker': 'o',
        'markerfacecolor': 'white',
        'markeredgecolor': 'black',
        'markersize': 10  # Marcador más grande
    },
    width=0.6  # Ancho de las cajas
)

# Personalización avanzada
# plt.title('Distribución del Uso de RAM por Threshold', fontsize=16, pad=20)
plt.xlabel('Threshold', fontsize=10)
plt.ylabel('Uso de RAM (mb)', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Usamos resources_stats para las anotaciones
for i, threshold in enumerate(ram_stats.index):
    mean_val = ram_stats.loc[threshold, 'mean']
    median_val = ram_stats.loc[threshold, '50%']
    std_val = ram_stats.loc[threshold, 'std']
    count = ram_stats.loc[threshold, 'count']
    
    plt.text(
        i, 
        nuevo_df['cpu_percent'].max() * 1.1,  # Posición arriba del máximo
        f"n={int(count)}\nμ={mean_val:.2f} ± {std_val:.2f}\nmediana={median_val:.2f}",
        ha='center',
        fontsize=8,
        bbox=dict(
            facecolor='white',
            alpha=0.8,
            edgecolor='gray',
            boxstyle='round,pad=0.5'
        )
    )

# Línea horizontal de referencia para CPU
# plt.axhline(
#     y=100, 
#     color='red', 
#     linestyle=':', 
#     alpha=0.5, 
#     label='Máximo teórico (RAM)'
# )
# plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

df_filtrado = nuevo_df[nuevo_df['device'] == 'cpu']
cpu_stats = df_filtrado.groupby('threshold')['cpu_percent'].describe()

plt.subplot(1, 2, i)  # Cambiado a 1, 2 porque solo hay dos columnas
sns.histplot(df_filtrado['cpu_percent'], kde=True, color='#ffb703', bins=15)

# Calcular estadísticas
mean_val = df_filtrado['cpu_percent'].mean()
median_val = df_filtrado['cpu_percent'].median()
std_val = df_filtrado['cpu_percent'].std()
count = len(df_filtrado['cpu_percent'])

# Añadir texto con estadísticas
plt.text(
    x=0.95, 
    y=0.95,
    s=f"n = {int(count)}\nμ = {mean_val:.2f}\nσ = {std_val:.2f}\nmed = {median_val:.2f}",
    transform=plt.gca().transAxes,
    ha='right',
    va='top',
    fontsize=10,
    bbox=dict(
        facecolor='white',
        alpha=0.8,
        edgecolor='gray',
        boxstyle='round,pad=0.5'
    )
)


plt.title('Uso de CPU', loc='center', fontsize=10)
plt.xlabel('Valor')
plt.ylabel('Frecuencia')
plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

df_filtrado = nuevo_df[nuevo_df['device'] == 'cpu']
cpu_stats = df_filtrado.groupby('threshold')['ram_mb'].describe()

plt.subplot(1, 2, i)  # Cambiado a 1, 2 porque solo hay dos columnas
sns.histplot(df_filtrado['ram_mb'], kde=True, color='#8ecae6', bins=15)

# Calcular estadísticas
mean_val = df_filtrado['ram_mb'].mean()
median_val = df_filtrado['ram_mb'].median()
std_val = df_filtrado['ram_mb'].std()
count = len(df_filtrado['ram_mb'])

# Añadir texto con estadísticas
plt.text(
    x=0.95, 
    y=0.95,
    s=f"n = {int(count)}\nμ = {mean_val:.2f}\nσ = {std_val:.2f}\nmed = {median_val:.2f}",
    transform=plt.gca().transAxes,
    ha='right',
    va='top',
    fontsize=10,
    bbox=dict(
        facecolor='white',
        alpha=0.8,
        edgecolor='gray',
        boxstyle='round,pad=0.5'
    )
)


plt.title('Uso de RAM', loc='center', fontsize=10)
plt.xlabel('Valor')
plt.ylabel('Frecuencia')
plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
nuevo_df.reset_index().plot(x='index', y=['cpu_percent', 'ram_mb'], 
                              style=['-o', '--s', ':^'],
                              markersize=8,
                              linewidth=2,
                              figsize=(12, 6))

plt.title('Evolución Temporal del Consumo', fontsize=14)
plt.xlabel('Muestra (index)')
plt.ylabel('Valor')
plt.legend(title='Métrica', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()